# Hyperbolic functions

This notebook covers `es.math.sinh`, `cosh`, `tanh` and their inverses `arcsinh`, `arccosh` and `arctanh`.
They take a dimensionless `Parameter` or `DescriptorNumber` and return a new one with the value, variance
and bounds worked out: the variance to first order, the bounds as the exact range of the function over the
bounds of the input.

The examples are taken from a diffraction experiment, but each one stands alone. Each part closes with a few exercises on what was just shown. Their hints and solutions are folded away and open on a click, so the text reads straight through; a solution is a worked answer with its result.

In [1]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Parameter

The function `show` prints a number as value, error and unit, and, for a `Parameter` with finite bounds, the bounds.

In [2]:
def show(number):
    """One line with value, error, unit and, for a Parameter, the finite bounds."""
    text = f'{number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min:.4f}, {number.max:.4f}]'
    return text

In [3]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## sinh, cosh and tanh

The three functions are monotonic or have one minimum, so their bounds are exact:

- `sinh` rises from $-\infty$ to $\infty$,
- `cosh` has its minimum 1 at 0 and is symmetric,
- `tanh` rises from $-1$ to $1$.

In [4]:
x = Parameter(name='x', value=0.8, variance=0.05**2, min=0.0, max=1.5)

for function in (es.math.sinh, es.math.cosh, es.math.tanh):
    print(f'{function.__name__:5}', show(function(x)))

sinh  0.8881 ± 0.0669 dimensionless   in [0.0000, 2.1293]
cosh  1.3374 ± 0.0444 dimensionless   in [1.0000, 2.3524]
tanh  0.6640 ± 0.0280 dimensionless   in [0.0000, 0.9051]


The magnetisation of a paramagnet is $M/M_s = \tanh(\mu B/k_B T)$ for spin one half. At a given field and
temperature the argument is a dimensionless number, so the unit arithmetic is done for you:

In [5]:
moment = Parameter(name='moment', value=0.5, variance=0.02**2, unit='meV/T')
field = DescriptorNumber(name='field', value=4.0, variance=0.0, unit='T')
thermal = DescriptorNumber(name='thermal', value=2.0, variance=0.0, unit='meV')

magnetisation = es.math.tanh(moment * field / thermal)
print(show(magnetisation))

0.7616 ± 0.0168 dimensionless   in [-1.0000, 1.0000]


### Exercise 1: Saturation of a magnet

A magnetisation follows $\tanh(x)$ with $x = 0.80 \pm 0.05$. The fitting range of $x$ is 0 to 3. What is the magnetisation, and between which values can it lie?

In [6]:
x = Parameter(name='x', value=0.80, variance=0.05**2, min=0.0, max=3.0)

<details class="question">
<summary>Hint 1</summary>

Call `es.math.tanh` on the parameter.

</details>

<details class="question">
<summary>Hint 2</summary>

`show` prints the bounds as well, because they are finite.

</details>

In [7]:
%%solution The upper bound is $\tanh(3)$, not 1: the bounds are the range over the bounds of the input.
magnetisation = es.math.tanh(x)
print(show(magnetisation))

## The inverse functions

`arcsinh` accepts any number, `arccosh` needs a number of at least 1, and `arctanh` a number between -1 and
1. The inverse of `tanh` turns a measured saturation back into the argument that produced it:

In [8]:
saturation = Parameter(name='saturation', value=0.5, variance=0.02**2, min=-0.9, max=0.9)
print(show(es.math.arctanh(saturation)))

y = Parameter(name='y', value=2.0, variance=0.05**2, min=1.0, max=4.0)
print(show(es.math.arccosh(y)))

0.5493 ± 0.0267 dimensionless   in [-1.4722, 1.4722]
1.3170 ± 0.0289 dimensionless   in [0.0000, 2.0634]


### Exercise 2: Recovering the argument

A magnet reaches $0.50 \pm 0.02$ of its saturation magnetisation, and the magnetisation follows $\tanh(x)$. What is $x$?

In [9]:
saturation = Parameter(name='saturation', value=0.50, variance=0.02**2)

<details class="question">
<summary>Hint 1</summary>

The inverse of `tanh` is `arctanh`.

</details>

<details class="question">
<summary>Hint 2</summary>

The derivative of `arctanh` is $1/(1 - y^2)$, so the error is $0.02/(1 - 0.25)$.

</details>

In [10]:
%%solution
x = es.math.arctanh(saturation)
print(show(x))

## Bounds that are not at the ends

The hyperbolic cosine has its minimum at 0, in the middle of an interval that contains 0. The lower bound
of the result is therefore 1, not the value at either end:

In [11]:
z = Parameter(name='z', value=0.0, variance=0.5**2, min=-1.0, max=1.0)
result = es.math.cosh(z)
print(show(result))

1.0000 ± 0.0000 dimensionless   in [1.0000, 1.5431]


### Exercise 3: The lowest value of cosh

A parameter can vary between -1 and 1, and its current value is 0.5 ± 0.05. What is the lowest value that `cosh` of it can take? Print only that number.

In [12]:
x = Parameter(name='x', value=0.5, variance=0.05**2, min=-1.0, max=1.0)

<details class="question">
<summary>Hint 1</summary>

The `Parameter` returned by `es.math.cosh` has the attributes `min` and `max`.

</details>

<details class="question">
<summary>Hint 2</summary>

Look at the lower bound, `result.min`, and compare it with $\cosh(-1) = 1.5431$.

</details>

In [13]:
%%solution The lower bound is 1, the value at 0, not $\cosh(-1)$.
result = es.math.cosh(x)
print(result.min)

## Values outside the domain

`arccosh` is only defined for numbers of at least 1. A value outside the domain is refused with an error
that names the parameter, as the next exercise shows.

### Exercise 4: Outside the domain

Predict what `es.math.arccosh` does with $x = 0.5 \pm 0.01$, then try it. Catch the error and print its message.

In [14]:
x = Parameter(name='x', value=0.5, variance=0.01**2)

<details class="question">
<summary>Hint 1</summary>

The domain of `arccosh` is $[1, \infty)$.

</details>

<details class="question">
<summary>Hint 2</summary>

Put the call in a `try` block and catch `ValueError`.

</details>

In [15]:
%%solution
try:
    es.math.arccosh(x)
except ValueError as error:
    print(error)

## A round trip

Each function undoes its inverse, with the value and the uncertainty unchanged:

In [16]:
x = Parameter(name='x', value=0.8, variance=0.05**2, min=0.0, max=1.5)
back = es.math.arcsinh(es.math.sinh(x))
print(show(back))

0.8000 ± 0.0500 dimensionless   in [0.0000, 1.5000]


## Charts

The magnetisation $\tanh(x)$ and its uncertainty for $x$ with an uncertainty of 0.2. The band is widest
where the curve is steepest, near 0, and vanishes as the curve saturates. The chart is interactive: hover
over the curve to read the values. It loads Plotly from the internet, so it needs a connection. The charts
are shown with a small helper, `show_figure(fig)`, which is defined in a hidden cell; in your own notebook
`fig.show()` does the same.

In [17]:
arguments = np.linspace(-3.0, 3.0, 61)

values, errors = [], []
for value in arguments:
    argument = Parameter(name='x', value=value, variance=0.2**2)
    magnetisation = es.math.tanh(argument)
    values.append(magnetisation.value)
    errors.append(magnetisation.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=arguments,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=arguments,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)

fig.add_trace(
    go.Scatter(
        x=arguments,
        y=values,
        mode='lines',
        name='tanh(x)',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='x = %{x:.2f}<br>tanh = %{y:.3f} ± %{customdata:.3f}<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='x',
    yaxis_title='tanh(x)',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.05, y=0.95),
)
show_figure(fig)

The inverse functions are the mirror image. The error of `arccosh` grows without limit at the edge of the
domain, $y = 1$, where the derivative $1/\sqrt{y^2 - 1}$ diverges:

In [18]:
ys = np.linspace(1.05, 5.0, 60)

values, errors = [], []
for value in ys:
    y = Parameter(name='y', value=value, variance=0.02**2)
    result = es.math.arccosh(y)
    values.append(result.value)
    errors.append(result.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ys,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=ys,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)

fig.add_trace(
    go.Scatter(
        x=ys,
        y=values,
        mode='lines',
        name='arccosh(y)',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='y = %{x:.2f}<br>arccosh = %{y:.3f} ± %{customdata:.3f}<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='y',
    yaxis_title='arccosh(y)',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.6, y=0.3),
)
show_figure(fig, height=380)

## Good to know

At the minimum of `cosh` the derivative is 0, so the first-order variance is 0, however large the
uncertainty of the input is. For an input $0 \pm 0.5$ the exact variance of $\cosh(x)$, with
$x \sim \mathcal{N}(0, 0.5^2)$, is $\tfrac{1}{2}(1 + e^{2\sigma^2}) - e^{\sigma^2} = 0.0403$:

In [19]:
flat = Parameter(name='flat', value=0.0, variance=0.5**2)
print(es.math.cosh(flat).variance)

sigma2 = 0.5**2
print(0.5 * (1 + np.exp(2 * sigma2)) - np.exp(sigma2))

0.0
0.0403352186623227


The same applies to every function at a point where its derivative vanishes. Large arguments overflow:
`sinh` of 800 is refused with an error instead of returning infinity.

In [20]:
try:
    es.math.sinh(Parameter(name='big', value=800.0, variance=1.0**2))
except ValueError as error:
    print(error)

sinh(big) is not finite: value=inf, variance=inf. Check the value and variance of the argument(s).
